In [6]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("../")
from utils.io import read_parameters

# =========================================================
# 1. READ PARAMETERS AND KEY FILE
# =========================================================

parameter_file = "../local_new/parameters_mosaic.yml"
parameters = read_parameters(parameter_file)

key_file_path = parameters["key_file"]
key_file = pd.read_csv(key_file_path)

print("key file columns:", key_file.columns)
print("condition_color levels:", key_file["condition_color"].unique())

output_folder = parameters["output_folder"]
data_folder = output_folder + "/tracking_data/"
summary_folder = output_folder + "/trajectory_summary_cumulative_mosaic"
os.makedirs(summary_folder, exist_ok=True)

key file columns: Index(['experimentID', 'filename', 'color', 'mosaic', 'treatment', 'condition',
       'date', 'condition_color'],
      dtype='object')
condition_color levels: ['mosaic_siCTRL_siCTRL_green' 'mosaic_siCTRL_siCTRL_red'
 'mosaic_siScr_siScr_green' 'mosaic_siScr_siScr_red'
 'mosaic_siCTRL_siCTRL_static_green' 'mosaic_siCTRL_siCTRL_static_red'
 'mosaic_siScr_siScr_static_green' 'mosaic_siScr_siScr_static_red']


In [10]:
# =========================================================
# 2. SETTINGS
# =========================================================

groups_to_analyze = [
    "mosaic_siCTRL_siCTRL_green",
    "mosaic_siScr_siScr_green",
    "mosaic_siCTRL_siCTRL_static_green",
    "mosaic_siScr_siScr_static_green"
    
]

# interval edges in hours relative to observation start
# intervals become:
# 0-5, 5-10, 10-20, 20-30, 30-40, 40-48
interval_edges_h = [0, 10, 20, 30, 40, 50, 60]

observation_time = parameters["observation_time"]
frames_per_hour = parameters["frames_per_hour"]

# tolerate small mismatch when looking for start/end frame of each interval
frame_tolerance_h = 1
frame_tolerance = int(frame_tolerance_h * frames_per_hour)

# if you want to subsample tracks per replicate, set to an integer like 100
# if you want to use all eligible tracks, set to None
tracks_per_replicate = None

random_seed = 42
rng = np.random.default_rng(random_seed)

In [11]:
# =========================================================
# 3. HELPER FUNCTION MOSAIC
#    CUMULATIVE SUM OF COVERAGE-WEIGHTED INTERVAL DISPLACEMENTS
# =========================================================

def summarize_tracking_file_cumulative_by_intervals(data, condition_color, replicate_id):
    """
    Build cumulative summaries by summing coverage-weighted interval mean displacements.

    Example:
    0-10h = weighted_mean(0-5h) + weighted_mean(5-10h)
    0-20h = weighted_mean(0-5h) + weighted_mean(5-10h) + weighted_mean(10-20h)

    For each interval:
    - include tracks that overlap the interval
    - compute displacement over the observed overlap
    - weight each track by overlap_frames / interval_frames
    """
    rows = []

    data = data.copy()
    data = data.sort_values(["TRACK_ID", "FRAME"])

    if data.empty:
        return rows

    interval_summaries = []

    for i in range(len(interval_edges_h) - 1):
        start_h = interval_edges_h[i]
        end_h = interval_edges_h[i + 1]

        target_start_frame = observation_time[0] + int(round(start_h * frames_per_hour))
        target_end_frame = observation_time[0] + int(round(end_h * frames_per_hour))

        interval_frame_count = target_end_frame - target_start_frame
        if interval_frame_count <= 0:
            continue

        track_rows = []

        for track_id, track_df in data.groupby("TRACK_ID"):
            track_df = track_df.sort_values("FRAME").copy()
            frames = track_df["FRAME"].values

            track_first = frames.min()
            track_last = frames.max()

            # overlap of this track with the target interval
            overlap_start = max(track_first, target_start_frame)
            overlap_end = min(track_last, target_end_frame)

            if overlap_end <= overlap_start:
                continue

            overlap_frames = overlap_end - overlap_start

            # require at least min_track_length frames inside the interval
            if overlap_frames < parameters["min_track_length"]:
                continue

            # choose actual start frame near overlap_start
            start_candidates = frames[np.abs(frames - overlap_start) <= frame_tolerance]
            if len(start_candidates) == 0:
                continue

            # choose actual end frame near overlap_end
            end_candidates = frames[np.abs(frames - overlap_end) <= frame_tolerance]
            if len(end_candidates) == 0:
                continue

            start_diffs = np.abs(start_candidates - overlap_start)
            actual_start_frame = start_candidates[start_diffs == start_diffs.min()].min()

            end_diffs = np.abs(end_candidates - overlap_end)
            actual_end_frame = end_candidates[end_diffs == end_diffs.min()].max()

            if actual_end_frame <= actual_start_frame:
                continue

            row_start = track_df[track_df["FRAME"] == actual_start_frame]
            row_end = track_df[track_df["FRAME"] == actual_end_frame]

            if row_start.empty or row_end.empty:
                continue

            start_x = row_start["POSITION_X"].iloc[0]
            start_y = row_start["POSITION_Y"].iloc[0]
            end_x = row_end["POSITION_X"].iloc[0]
            end_y = row_end["POSITION_Y"].iloc[0]

            delta_x = end_x - start_x
            delta_y = end_y - start_y
            delta_d = np.sqrt(delta_x**2 + delta_y**2)

            coverage_weight = (actual_end_frame - actual_start_frame) / interval_frame_count

            track_rows.append({
                "TRACK_ID": track_id,
                "actual_start_frame": actual_start_frame,
                "actual_end_frame": actual_end_frame,
                "overlap_frames": actual_end_frame - actual_start_frame,
                "interval_frame_count": interval_frame_count,
                "coverage_weight": coverage_weight,
                "delta_x": delta_x,
                "delta_y": delta_y,
                "delta_d": delta_d,
            })

        if len(track_rows) == 0:
            interval_summaries.append({
                "start_h": start_h,
                "end_h": end_h,
                "n_tracks_used_interval": 0,
                "sum_weights": np.nan,
                "weighted_mean_interval_dx": np.nan,
                "weighted_mean_interval_dy": np.nan,
                "weighted_mean_interval_d": np.nan,
            })
        else:
            track_df = pd.DataFrame(track_rows)

            # optional subsampling after eligible interval tracks are defined
            if tracks_per_replicate is not None:
                n_sample = min(tracks_per_replicate, len(track_df))
                track_df = track_df.sample(n=n_sample, random_state=random_seed)

            w = track_df["coverage_weight"].values
            sum_w = w.sum()

            weighted_mean_dx = np.sum(track_df["delta_x"].values * w) / sum_w
            weighted_mean_dy = np.sum(track_df["delta_y"].values * w) / sum_w
            weighted_mean_d = np.sum(track_df["delta_d"].values * w) / sum_w

            interval_summaries.append({
                "start_h": start_h,
                "end_h": end_h,
                "n_tracks_used_interval": len(track_df),
                "sum_weights": sum_w,
                "weighted_mean_interval_dx": weighted_mean_dx,
                "weighted_mean_interval_dy": weighted_mean_dy,
                "weighted_mean_interval_d": weighted_mean_d,
            })

    interval_df = pd.DataFrame(interval_summaries)

    cum_dx = 0.0
    cum_dy = 0.0
    cum_d = 0.0

    for _, r in interval_df.iterrows():
        if pd.isna(r["weighted_mean_interval_dx"]) or pd.isna(r["weighted_mean_interval_d"]):
            continue

        cum_dx += r["weighted_mean_interval_dx"]
        cum_dy += r["weighted_mean_interval_dy"]
        cum_d += r["weighted_mean_interval_d"]

        rows.append({
            "condition_color": condition_color,
            "replicate": replicate_id,
            "time_window": f"0-{int(r['end_h'])}h",
            "time_end_h": r["end_h"],
            "n_tracks_used_interval": r["n_tracks_used_interval"],
            "sum_weights": r["sum_weights"],
            "weighted_mean_interval_dx": r["weighted_mean_interval_dx"],
            "weighted_mean_interval_dy": r["weighted_mean_interval_dy"],
            "weighted_mean_interval_d": r["weighted_mean_interval_d"],
            "mean_cum_dx": cum_dx,
            "mean_cum_dy": cum_dy,
            "mean_cum_d": cum_d,
        })

    return rows

In [12]:
# =========================================================
# 4. BUILD REPLICATE-LEVEL CUMULATIVE SUMMARY TABLE MOSAIC
# =========================================================

summary_rows = []

for group in groups_to_analyze:
    print(f"\nProcessing group: {group}")

    key_select = key_file[key_file["condition_color"] == group].copy()

    if key_select.empty:
        print(f"No key file rows found for condition_color = {group}")
        continue

    for _, row in key_select.iterrows():
        treatment = str(row["treatment"]).strip()
        color = str(row["color"]).strip()
        replicate_id = str(row["experimentID"]).strip()

        tracking_file = f"tracking_data_{treatment}_{color}_{replicate_id}.csv"
        tracking_path = data_folder + tracking_file

        if not os.path.exists(tracking_path):
            print("Missing file:", tracking_path)
            continue

        data = pd.read_csv(tracking_path, low_memory=False)

        for col in ["FRAME", "POSITION_X", "POSITION_Y", "TRACK_ID"]:
            data[col] = pd.to_numeric(data[col], errors="coerce")

        data = data.dropna(subset=["FRAME", "POSITION_X", "POSITION_Y", "TRACK_ID"]).copy()
        data["FRAME"] = data["FRAME"].astype(int)
        data["TRACK_ID"] = data["TRACK_ID"].astype(int)

        result_rows = summarize_tracking_file_cumulative_by_intervals(
            data=data,
            condition_color=group,
            replicate_id=replicate_id
        )

        if len(result_rows) > 0:
            summary_rows.extend(result_rows)

summary_df = pd.DataFrame(summary_rows)

if not summary_df.empty:
    summary_df["condition_color"] = pd.Categorical(
        summary_df["condition_color"].astype(str).str.strip(),
        categories=groups_to_analyze,
        ordered=True
    )
    summary_df["time_window"] = pd.Categorical(
        summary_df["time_window"],
        categories=[f"0-{x}h" for x in interval_edges_h[1:]],
        ordered=True
    )

print("\nReplicate-level cumulative trajectory summary:")
display(summary_df)

summary_outfile = os.path.join(summary_folder, "replicate_level_cumulative_trajectory_summary_mosaic_green.csv")
summary_df.to_csv(summary_outfile, index=False)

print("Saved:", summary_outfile)


Processing group: mosaic_siCTRL_siCTRL_green

Processing group: mosaic_siScr_siScr_green

Processing group: mosaic_siCTRL_siCTRL_static_green

Processing group: mosaic_siScr_siScr_static_green

Replicate-level cumulative trajectory summary:


,condition_color,replicate,time_window,time_end_h,n_tracks_used_interval,sum_weights,weighted_mean_interval_dx,weighted_mean_interval_dy,weighted_mean_interval_d,mean_cum_dx,mean_cum_dy,mean_cum_d
0,mosaic_siCTRL_siCTRL_green,250926_siCTRL_mosaic_5050_green,0-10h,10.0,966.0,418.2125,-17.530197,-9.221403,49.784468,-17.530197,-9.221403,49.784468
1,mosaic_siCTRL_siCTRL_green,250926_siCTRL_mosaic_5050_green,0-20h,20.0,840.0,405.7500,-12.876859,-4.751352,37.847669,-30.407056,-13.972755,87.632137
2,mosaic_siCTRL_siCTRL_green,250926_siCTRL_mosaic_5050_green,0-30h,30.0,788.0,378.3750,-12.942852,-3.857464,30.210373,-43.349908,-17.830219,117.842510
3,mosaic_siCTRL_siCTRL_green,250926_siCTRL_mosaic_5050_green,0-40h,40.0,704.0,360.6000,-12.753367,-0.894974,27.003560,-56.103275,-18.725193,144.846071
4,mosaic_siCTRL_siCTRL_green,250926_siCTRL_mosaic_5050_green,0-50h,50.0,658.0,356.9750,-19.091007,-5.536952,29.782151,-75.194282,-24.262146,174.628222
5,mosaic_siCTRL_siCTRL_green,250926_siCTRL_mosaic_5050_green,0-60h,60.0,457.0,214.5000,-18.695402,-5.043639,24.957610,-93.889684,-29.305784,199.585832
6,mosaic_siScr_siScr_green,250926_siScr_mosaic_7030_green,0-10h,10.0,1511.0,627.3625,-31.223280,-3.897032,56.382163,-31.223280,-3.897032,56.382163
7,mosaic_siScr_siScr_green,250926_siScr_mosaic_7030_green,0-20h,20.0,1384.0,667.0750,-17.851336,2.034715,37.350726,-49.074616,-1.862317,93.732889
8,mosaic_siScr_siScr_green,250926_siScr_mosaic_7030_green,0-30h,30.0,1401.0,629.2000,-13.532430,1.168026,29.886388,-62.607046,-0.694291,123.619276
9,mosaic_siScr_siScr_green,250926_siScr_mosaic_7030_green,0-40h,40.0,1248.0,596.8875,-18.143315,-1.794464,29.396543,-80.750362,-2.488755,153.015819


Saved: T:/Olya/Data/analysis/merged_time_corrected/061026/output//trajectory_summary_cumulative_mosaic\replicate_level_cumulative_trajectory_summary_mosaic_green.csv
